# Laya run 5: Passages on the v1 envelope — Verdict only (Kaggle, GPU T4 ×2)

Fifth run, testing **Passages (Fix 1) on the canonical v1 envelope**.

**Why this run exists:**
* **Run 2** showed the model had two issues:
  1. Short findings are biased toward "not a quote" (coincidence).
  2. Consecutive verses are treated independently, occasionally dropping a verse in the middle of a recitation chain (e.g. الواقعة 89).
* **Run 3 & Run 4** proved that merging consecutive verses into passages (D) **completely fixed the middle-of-chain drop** (0/64 real verses missed vs 3/64 before).
* **BUT Run 4 also revealed why general verdict dropped:**
  When we removed all boundary questions (`both_verdict`), the AUC stayed low (**0.836**), while `v1_verdict` (the old v1 envelope) kept **0.933 AUC / 91.4% accuracy / 4.1% real rejected**.
  The culprit was the **v2 envelope format itself**: pushing `canonical_corpus_text` to the very bottom behind hundreds of tokens of transcript, and adding noisy, heavily skewed `neighbor_findings` JSON blobs.
* **The fix tested here:** Keep the exact proven **v1 envelope** (`canonical_corpus_text` right next to `candidate_reference`), merge consecutive verses into a single passage, drop noisy neighbours, and train **verdict only**!

| run | data | envelope | questions | what it answers |
|---|---|---|---|---|
| `v1_passages` | `v1_passages` (merged passages) | **v1 format** (corpus text at top) | **verdict only** | **The candidate:** Does adding passages to the v1 envelope fix chains while preserving ~0.94+ AUC? |
| `passages_verdict` | `passages` (merged passages) | **v2 format** (corpus text at bottom) | **verdict only** | **The control:** Does moving corpus text to the top make a measurable difference on identical passage data? |

**Benchmarks to beat:**
* `before: soft_asis` (production model): 91.4% accuracy, 0.944 AUC, 6.4% real rejected.
* Run 2 `curve_100`: 89.4% accuracy, 0.945 AUC, 85.2% not-a-quote recall.
* Run 4 `v1_verdict`: 91.4% accuracy, 0.933 AUC, 4.1% real rejected.

### Kaggle settings (right-hand panel)
* **Accelerator:** `GPU T4 x2` · **Internet:** `On` · **Secrets:** `HF_TOKEN`
* **Input:** the `laya_v2` dataset (works with the existing version or the new zip!)
* **Run:** *Save Version → Save & Run All (Commit)*. Expect **~25–30 min** total.
* **Download afterwards:** `results/` folder.

## 1. Environment & dual T4 GPU check

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}! Notebook options -> Accelerator -> GPU T4 x2."
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Install dependencies

In [ ]:
# pinned to the 0.4 line the Wahy data was validated with (laya.train / laya.calibrate are identical 0.4.12 -> 0.4.15)
!pip install -q "laya>=0.4.12,<0.5" transformers torch packaging accelerate

## 3. Dataset, scripts, base checkpoint, Hub login

In [ ]:
import glob, json, sys
from pathlib import Path
from huggingface_hub import snapshot_download, login
from laya.agent import _fix_tokenizer_config

DATA = next(Path(p).parent for p in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
            if json.loads(Path(p).read_text()).get("name") == "laya_v2")
manifest = json.loads((DATA / "manifest.json").read_text())
sys.path.insert(0, str(DATA))
import eval_laya

print("dataset:", DATA, "| created", manifest.get("created"), "| base model", manifest.get("base_model"))

WORK = Path("/kaggle/working")
for d in ("results", "logs", "data"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

# Build v1_passages if not already provided in DATA
V1_PASS_DIR = DATA / "v1_passages" if (DATA / "v1_passages").exists() else (WORK / "data" / "v1_passages")
if not (V1_PASS_DIR / "train.jsonl").exists():
    print("Converting DATA/passages to v1 envelope format (corpus text at top, verdict only)...")
    V1_PASS_DIR.mkdir(parents=True, exist_ok=True)
    v1_instructions = (
        "In this Arabic Islamic lecture transcript, determine whether `matched_span_in_lecture` "
        "(surrounded by `context_before` and `context_after`) is an actual quotation/citation of "
        "`canonical_corpus_text` (`source_type`), or if it is merely ordinary speaker prose/explanation "
        "that happens to share words with the corpus text."
    )
    v1_criteria = {
        "CORRECT": "The speaker is genuinely quoting, reciting, or invoking this Quranic Ayah, Prophetic Hadith, or Dhikr (verbatim or as a direct religious citation in context).",
        "FALSE": "False positive: the words are ordinary conversational speech, fiqh explanation, or narrator/person names that coincidentally overlap with corpus vocabulary without quoting the text.",
        "NOT_SURE": "Ambiguous or partial paraphrase/meaning-only allusion where it is borderline whether the speaker intended a direct quotation."
    }
    for split in ("train", "val", "test"):
        in_path = DATA / "passages" / f"{split}.jsonl"
        out_path = V1_PASS_DIR / f"{split}.jsonl"
        with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
            for line in fin:
                r = json.loads(line)
                st = r["state"]
                v1_state = {
                    "source_type": st.get("source_type"),
                    "candidate_reference": st.get("candidate_reference"),
                    "canonical_corpus_text": st.get("canonical_corpus_text"),
                    "context_before": st.get("context_before"),
                    "matched_span_in_lecture": st.get("matched_span_in_lecture"),
                    "context_after": st.get("context_after"),
                    "unmatched_corpus_prefix_words": st.get("unmatched_corpus_prefix_words", ""),
                    "immediate_transcript_before": st.get("immediate_transcript_before", ""),
                    "unmatched_corpus_suffix_words": st.get("unmatched_corpus_suffix_words", ""),
                    "immediate_transcript_after": st.get("immediate_transcript_after", ""),
                    "quran_neighboring_context": st.get("quran_neighboring_context"),
                    "has_citation_cue": bool(st.get("has_citation_cue", False))
                }
                v1_row = {
                    "id": r["id"],
                    "state": v1_state,
                    "questions": {
                        "verdict": {
                            "type": "choice",
                            "instructions": v1_instructions,
                            "criteria": v1_criteria
                        }
                    },
                    "gold": {"verdict": r["gold"]["verdict"]},
                    "meta": r["meta"]
                }
                fout.write(json.dumps(v1_row, ensure_ascii=False) + "\n")
        print(f"  {split}: {out_path.stat().st_size / 1e3:.0f} KB written")

MODEL_ID = manifest["base_model"]
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
print("base checkpoint:", model_dir)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Hub: logged in")
except Exception as e:
    HF_TOKEN = None
    print("Hub: no HF_TOKEN secret:", e)

## 4. Settings
Both models train on **Verdict Only** with 2 epochs, effective batch 64 (4 × 8 accumulation), max_len 2048, and RLCD loss.

In [ ]:
RUNS = {
    "v1_passages": {
        "data_dir": V1_PASS_DIR,
        "questions": ["verdict"],
        "cfg_extra": {"wahy_envelope": 1, "wahy_envelope_options": {"passages": True}, "wahy_questions": ["verdict"]},
        "about": "v1 envelope (corpus text at top) + passages, verdict only"
    },
    "passages_verdict": {
        "data_dir": DATA / "passages",
        "questions": ["verdict"],
        "cfg_extra": {"wahy_envelope": 2, "wahy_envelope_options": {"passages": True}, "wahy_questions": ["verdict"]},
        "about": "v2 envelope (corpus text at bottom) + passages, verdict only"
    }
}

TRAIN_ARGS = (f"--epochs 2 --micro-batch 4 --grad-accum 8 --lr-encoder 2.5e-5 --lr-head 1e-4 --loss rlcd "
              f"--max-len 2048 --head-max-len 256")
APP_FALSE_SHARE = 0.10
BEFORE_MODEL = "ZenoAbderrahim/laya-wahy-soft_asis"
PUSH_TO_HUB = True
HUB_PREFIX = "ZenoAbderrahim/laya-wahy-r5"

def run(cmd, log):
    print("$", cmd, flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if "Warning" not in line and "warn(" not in line:
                print(line, end="", flush=True)
        return p.wait()

import pandas as pd
rows = []
for name, cfg in RUNS.items():
    tr_lines = [json.loads(l) for l in open(cfg["data_dir"] / "train.jsonl")]
    n_rows = len(tr_lines)
    n_false = sum(r["meta"]["verdict"] == "FALSE" for r in tr_lines)
    rows.append({
        "run": name,
        "questions": ", ".join(cfg["questions"]),
        "about": cfg["about"],
        "train rows": n_rows,
        "not a quote (share)": round(n_false / n_rows, 3)
    })
pd.DataFrame(rows).set_index("run")

## 5. The "before" line: current production model

In [ ]:
flags = f"--verse-flags {DATA}/test_verse_flags.json"
if HF_TOKEN:
    rc = run(f"python {DATA}/eval_laya.py predict --model {BEFORE_MODEL} --name 'before: soft_asis (app)' "
             f"--test {DATA}/v1_test.jsonl --out {WORK}/results/before_soft_asis.json --app-false-share {APP_FALSE_SHARE} "
             f"--batch-size 32 --baselines {DATA}/v1_baselines.json {flags}",
             WORK / "logs" / "eval_before.log")
    print("before:", "ok" if rc == 0 else f"failed (exit {rc}), see logs/eval_before.log")

## 6. Train + evaluate

In [ ]:
import time
status = {}

for name, cfg in RUNS.items():
    data_dir = cfg["data_dir"]
    qs = cfg["questions"]
    out = WORK / "models" / name
    t0 = time.time()
    try:
        cfg_extra_clean = json.dumps(cfg["cfg_extra"]).replace('"', '\\"')
        qarg = f"--questions {','.join(qs)}" if qs else ""
        rc = run(f"torchrun --standalone --nproc_per_node=2 {DATA}/train_ddp.py {model_dir} {out} "
                 f"--train {data_dir}/train.jsonl --val {data_dir}/val.jsonl --name laya-wahy-r5-{name} "
                 f"--cfg-extra \"{cfg_extra_clean}\" {qarg} {TRAIN_ARGS}", WORK / "logs" / f"train_{name}.log")
        if rc != 0 or not (out / "model.safetensors").exists():
            status[name] = f"training failed (exit {rc}), see logs/train_{name}.log"
            continue
        
        tr_lines = [json.loads(l) for l in open(data_dir / "train.jsonl")]
        tr_false_share = round(sum(r["meta"]["verdict"] == "FALSE" for r in tr_lines) / len(tr_lines), 4)
        
        rc = run(f"python {DATA}/eval_laya.py predict --model {out} --name {name} --test {data_dir}/test.jsonl "
                 f"--out {WORK}/results/{name}.json --train-false-share {tr_false_share} "
                 f"--app-false-share {APP_FALSE_SHARE} --batch-size 16 {flags}", WORK / "logs" / f"eval_{name}.log")
        status[name] = f"ok in {(time.time() - t0) / 60:.0f} min" if rc == 0 else f"evaluation failed (exit {rc})"
        train_log = out / "train_log.json"
        if train_log.exists():
            (WORK / "results" / f"train_log_{name}.json").write_text(train_log.read_text())
    except Exception as e:
        status[name] = f"error: {e}"
    print(f"\n##### {name}: {status[name]}\n", flush=True)

json.dump(status, open(WORK / "results" / "status.json", "w"), indent=1)
status

## 7. Load results and benchmarks

In [ ]:
from IPython.display import Markdown, display

v1_rows = eval_laya.read_jsonl(DATA / "v1_test.jsonl")
vflags = json.loads((DATA / "test_verse_flags.json").read_text())
res = {n: json.load(open(WORK / "results" / f"{n}.json")) for n in RUNS if (WORK / "results" / f"{n}.json").exists()}
before = [json.load(open(WORK / "results" / "before_soft_asis.json"))] if (WORK / "results" / "before_soft_asis.json").exists() else []
teachers = [json.load(open(f)) for f in sorted(glob.glob(str(WORK / "results" / "baseline_*.json")))]
models = before + [res[n] for n in RUNS if n in res]

print("Models loaded:", [m["name"] for m in models])

## 8. Verdict evaluation (Overall & Context Slices)

In [ ]:
report = []
for g in ["you", "you + Sonnet", "Sonnet · other lectures", "written text"]:
    t = f"### verdict · {g}\n" + eval_laya.compare(models + teachers, g)
    report.append(t); display(Markdown(t))

for g in ["you", "you + Sonnet"]:
    for part in ("by context", "by span length"):
        t = f"### verdict {part} · {g}\n" + eval_laya.compare_slices(models, g, part)
        report.append(t); display(Markdown(t))

(WORK / "results" / "verdict_report.md").write_text("\n\n".join(report), encoding="utf-8")

## 9. Decision rule & comparison against benchmarks

In [ ]:
print("=== BENCHMARK COMPARISON (Cards: you) ===")
print("Run 1 (soft_asis production):  AUC 0.944 | Acc 91.4% | Real rejected 6.4%")
print("Run 2 (curve_100):             AUC 0.945 | Acc 89.4% | Real rejected 9.9%")
print("Run 4 (v1_verdict):            AUC 0.933 | Acc 91.4% | Real rejected 4.1%")
print("Run 4 (both_verdict v2 D+A):   AUC 0.836 | Acc 83.3% | Real rejected 12.3%")
print("========================================\n")

if "v1_passages" in res:
    r_you = res["v1_passages"]["groups"]["you"]["verdict"]
    auc_val = r_you["auc"]
    acc_val = r_you["accuracy"][0]
    rej_val = r_you["real quotes wrongly rejected"][0]
    print(f"v1_passages result on 'you': AUC {auc_val:.3f} | Acc {acc_val:.1f}% | Real rejected {rej_val:.1f}%")
    if auc_val >= 0.94 and acc_val >= 90.0:
        print("\nSUCCESS: v1_passages matched/beat the ~0.94 benchmark while keeping passage merging!")
        print("-> Ready to deploy in production with word-overlap boundary rule.")
    else:
        print(f"\nNotice: AUC is {auc_val:.3f}. Compare against passages_verdict and v1_verdict.")

## 10. Visual comparison

In [ ]:
import matplotlib.pyplot as plt

names = [m["name"] for m in models]
aucs = [m["groups"].get("you", {}).get("verdict", {}).get("auc", 0) for m in models]
accs = [(m["groups"].get("you", {}).get("verdict", {}).get("accuracy", [0])[0]) for m in models]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.bar(names, aucs, color="skyblue")
ax1.axhline(0.945, color="red", linestyle="--", label="Run 2 benchmark (0.945)")
ax1.set_title("Verdict AUC (Cards: you)")
ax1.set_ylim(0.75, 1.0)
ax1.legend()

ax2.bar(names, accs, color="salmon")
ax2.axhline(91.4, color="red", linestyle="--", label="soft_asis (91.4%)")
ax2.set_title("Verdict Accuracy % (Cards: you)")
ax2.set_ylim(70, 100)
ax2.legend()

plt.tight_layout()
plt.savefig(WORK / "results" / "comparison_r5.png")
plt.show()

## 11. (Optional) Push models to Hugging Face Hub (Private)

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for n in RUNS:
        out = WORK / "models" / n
        if not (out / "model.safetensors").exists():
            continue
        repo_id = f"{HUB_PREFIX}-{n}"
        api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
        api.upload_folder(folder_path=str(out), repo_id=repo_id)
        print(f"pushed {repo_id}")